# Strategic Blocking: frozen transfer evaluation
Run the cells in order in a fresh GPU runtime. This performs inference only for base / before / after policies, using a separate Transformers/PEFT protocol. It does not reproduce local RAGEN sampling. All code and documentation are English.

Put `colab-transfer-v1.zip` in `MyDrive/strategic-blocking-transfer/`. Results are saved to Drive. No Colab GPU run has yet been validated; the CPU helpers and fixture rules have been tested.

In [ ]:
%pip install transformers==4.56.1 peft==0.21.2 accelerate==1.15.0 safetensors==0.8.0 pyyaml==6.0.3


If these libraries were already imported, restart the runtime after installation and continue below. Keep Colab's existing CUDA-enabled Torch; do not install the Windows/WSL environment wholesale.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import zipfile, tempfile, json, hashlib, sys, subprocess
archive = Path('/content/drive/MyDrive/strategic-blocking-transfer/colab-transfer-v1.zip')
if not archive.is_file():
    raise FileNotFoundError(f'Upload the handoff ZIP first: {archive}')
bundle = Path(tempfile.mkdtemp(prefix='strategic-blocking-', dir='/content'))
with zipfile.ZipFile(archive) as z:
    for item in z.infolist():
        target = (bundle / item.filename).resolve()
        if not target.is_relative_to(bundle.resolve()):
            raise ValueError('Unsafe ZIP path')
    z.extractall(bundle)
manifest = json.loads((bundle / 'bundle-manifest.json').read_text())
for name, digest in manifest.items():
    target = (bundle / name).resolve()
    if not target.is_relative_to(bundle.resolve()):
        raise ValueError('Unsafe manifest path')
    assert hashlib.sha256(target.read_bytes()).hexdigest() == digest, name
print('Verified bundle:', bundle)
output_parent = Path('/content/drive/MyDrive/strategic-blocking-transfer/results')
runner = bundle / 'scripts/evaluate_colab_transfer.py'
def run(*arguments):
    subprocess.run([sys.executable, '-u', str(runner), '--bundle', str(bundle),
                    '--output-parent', str(output_parent), *arguments], check=True)
run('--verify-only')


In [ ]:
subprocess.run([sys.executable, str(bundle / 'scripts/validate_frozen_suite.py')], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime and reconnect.'
print('GPU:', torch.cuda.get_device_name(), 'Torch:', torch.__version__)
run('--smoke')


## Frozen evaluation
Proceed only after the smoke test finishes successfully for all three policies. This runs 8 maps × 50 episodes × 3 policies = 1,200 episodes. Do not alter maps, prompt, sampling or seed after seeing results. A completed run has a `COMPLETE` file. Partial runs are not final comparisons.

In [ ]:
run()


## Return the evidence
Share the full completed `colab-transfer-*` result folder with YC: manifest, dependencies, raw episodes, summary, comparison, and COMPLETE. Report all maps, including failures. If the runtime stops, preserve the partial folder and error; do not silently pool partial and restarted runs.